# PGAT-FiLM pepper leaf disease experiments (Kaggle)

Settings: Accelerator = GPU T4, Internet = On.
Inputs: `arjuntejaswi/plant-village`, `udi17live/black-pepper-leaf-blight-and-yellow-mottle-virus`, and this code as a dataset.

Set `DATASET` (`bell` / `black`) and `PART` (`all`, `ours`, `baselines_a`, `baselines_b`) below, then run all cells.
The results are written to a zip file in the Output tab.

In [ ]:
# ===== SETTINGS - change these two lines for each run =====
DATASET = "black"     # "bell" or "black"
PART    = "all"       # "all", "ours", "baselines_a", "baselines_b", or "quick" (5-minute test)
CLASS_WEIGHT = True   # balances the loss when classes have different sizes


In [ ]:
import os, glob, shutil, subprocess, sys, zipfile
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q imagehash timm fvcore tabulate
import torch; print("torch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: set Accelerator = GPU T4 x2 in Session options"

# copy the code from the uploaded code dataset
hits = glob.glob("/kaggle/input/**/run_all.py", recursive=True)
assert hits, "Code not found: add your uploaded pepper_revision_code dataset as an input"
src = os.path.dirname(hits[0])
for f in os.listdir(src):
    if f.endswith((".py", ".txt", ".md")):
        shutil.copy(os.path.join(src, f), "/kaggle/working/")
os.chdir("/kaggle/working"); print("code copied from", src)


In [ ]:
# ===== find the images and build one folder per class (files are linked, not copied) =====
IMG = (".jpg", ".jpeg", ".png", ".bmp")
def image_dirs(root):
    out = {}
    for d, _, files in os.walk(root):
        n = sum(f.lower().endswith(IMG) for f in files)
        if n: out[d] = n
    return out

all_dirs = image_dirs("/kaggle/input")
if DATASET == "bell":
    wanted = {"Pepper__bell___Bacterial_spot": "bacterial_spot", "Pepper__bell___healthy": "healthy"}
    class_dirs = {}
    for d in all_dirs:
        b = os.path.basename(d)
        if b in wanted:
            class_dirs.setdefault(wanted[b], []).append(d)
else:
    skip = ("pepper__bell", "plantvillage", "tomato", "potato")
    class_dirs = {}
    for d in all_dirs:
        if any(s in d.lower() for s in skip) or "black" not in d.lower() and "pepper" not in d.lower():
            continue
        if any(o.startswith(d + os.sep) for o in all_dirs):
            continue                          # a parent folder of class folders, not a class
        name = os.path.basename(d).lower().strip()
        if name in ("train", "test", "val", "valid", "validation", "images"):
            continue
        class_dirs.setdefault(name, []).append(d)

print("Class folders found:")
for c, ds in class_dirs.items():
    print(f"  {c}: {sum(all_dirs[d] for d in ds)} images from {ds}")
# If the black-pepper classes are wrong, edit by hand, e.g.
# class_dirs = {"leaf_blight": ["/kaggle/input/.../black_pepper_leaf_blight"], "yellow_mottle_virus": [...], "healthy": [...]}
assert len(class_dirs) >= 2, "Fewer than 2 classes found - set class_dirs by hand (see comment above)"

DATA = f"/kaggle/working/data/{DATASET}"
shutil.rmtree(DATA, ignore_errors=True)
for c, ds in class_dirs.items():
    os.makedirs(f"{DATA}/{c}", exist_ok=True)
    k = 0
    for d in sorted(ds):
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(IMG):
                os.symlink(os.path.join(d, f), f"{DATA}/{c}/{k:05d}_{f}"); k += 1
print("data folder ready:", DATA)


In [ ]:
# ===== run =====
OURS = ["proposed", "abl_backbone_only", "abl_no_film", "abl_no_graph", "abl_grid_only", "abl_knn_only"]
PARTS = {
    "ours": OURS,
    "baselines_a": ["resnet50", "vgg16_bn", "densenet121", "inception_v3", "efficientnet_b0"],
    "baselines_b": ["mobilenetv2_100", "mobilenetv3_large_100", "vit_small_patch16_224",
                    "deit_small_patch16_224", "swin_tiny_patch4_window7_224"],
}
PARTS["all"] = PARTS["ours"] + PARTS["baselines_a"] + PARTS["baselines_b"]

cmd = [sys.executable, "run_all.py", "--data", DATA, "--name", DATASET, "--workers", "4"]
if CLASS_WEIGHT: cmd.append("--class_weight")
if PART == "quick":
    cmd += ["--quick", "--models", "proposed", "resnet50"]
else:
    cmd += ["--models"] + PARTS[PART]
print(" ".join(cmd))
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    if " ep " not in line or line.strip().startswith("ep   1") or "ep  30" in line:
        print(line, end="")
p.wait(); print("exit code", p.returncode)


In [ ]:
# ===== package results for download =====
res = f"results/{DATASET}" + ("_quick" if PART == "quick" else "")
out = f"/kaggle/working/{DATASET}_{PART}_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(res):
        for f in files:
            if f.endswith(".pt") and "proposed" not in root:
                continue                      # skip large baseline weights
            z.write(os.path.join(root, f))
    for f in glob.glob(f"splits/{DATASET}_*.csv"):
        z.write(f)
print("download this file from the Output tab:", out, round(os.path.getsize(out) / 1e6, 1), "MB")
if os.path.exists(f"{res}/tables.md"):
    print(open(f"{res}/tables.md").read())
